# Metric Definition and Product Sense — Measuring What Matters

**Author:** Siddharth Bokka  
**Project:** FlowDesk — B2B SaaS Product Analytics Portfolio  
**Notebook:** 05 of 07

---

> **Interview Context:** This notebook simulates the **Meta Analytical Reasoning round** and **Airbnb metric design rounds**.
> The core question: *"Given a new product feature, how would you define success?"*
>
> This is one of the highest-signal questions in product DS interviews. The interviewer is not looking for a single metric — they are evaluating whether you think in systems, understand tradeoffs, and can connect product decisions to measurable outcomes.

---

## Sections
1. The Metric Framework
2. Feature 1 — Smart Daily Digest (AI email summary)
3. Feature 2 — Project Templates (one-click setup)
4. Feature 3 — Client Portal (external stakeholder access)
5. DAU/MAU Ratio — Product Stickiness Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import scipy.stats as stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
    'figure.figsize': (10, 5),
})
sns.set_palette('tab10')

users        = pd.read_parquet('../data/users.parquet')
events       = pd.read_parquet('../data/events.parquet')
experiments  = pd.read_parquet('../data/experiments.parquet')
workspaces   = pd.read_parquet('../data/workspaces.parquet')
tickets      = pd.read_parquet('../data/support_tickets.parquet')

users['signup_date'] = pd.to_datetime(users['signup_date'])
events['event_ts']   = pd.to_datetime(events['event_ts'])
tickets['created_at']= pd.to_datetime(tickets['created_at'])

print(f"Users: {len(users):,} | Events: {len(events):,}")

---
## Section 1 — The Metric Framework

Before picking a metric, top product data scientists use a structured framework to ensure they are measuring the **right thing**, not just the **easy thing**. Here is the five-part framework used at Meta, Airbnb, and Stripe:

---

### 1. Primary Metric
The **single number** that captures whether the feature succeeded. It must be:
- **Measurable** — you can compute it from logged data
- **Attributable** — you can isolate the feature's contribution (ideally via A/B test)
- **Sensitive** — it moves enough within your experiment timeline to be detected
- **Aligned with business value** — improving it actually means the product is better

> *Rule of thumb: If you can only show your PM one number, what would it be?*

---

### 2. Guardrail Metrics
Things that **must NOT get worse** when the feature improves the primary metric. Guardrail metrics protect against:
- Optimizing one thing at the expense of another
- Short-term gains that cause long-term harm
- User experience degradation in adjacent flows

> *Example: A feature that increases DAU by showing more notifications might also increase unsubscribe rate — guardrails catch this.*

---

### 3. Counter-Metrics / Negative Signals
**Active signals of harm** — user actions that indicate the feature is creating friction, distrust, or frustration. Different from guardrails in that these are *intentional user pushback*, not just correlated declines.

> *Examples: hide post, block user, cancel subscription, mark as spam, submit complaint ticket*

---

### 4. Leading Indicators
**Proxy metrics** that move *faster* than the primary metric. Since primary metrics often require 2-4 weeks of experiment data, leading indicators give early signal.

> *Example: "email open rate" is a leading indicator for "in-app session from email" — it moves in 24h instead of 7 days.*

---

### 5. Ecosystem Metrics
How does this feature affect **other parts of the product**? Features rarely exist in isolation — they can cannibalize, complement, or redirect user behavior elsewhere.

> *Example: An AI summary email might reduce in-app notification clicks — not necessarily bad, but important to track.*

---

### Why This Framework Matters in Interviews

Interviewers are testing whether you:
1. Know the difference between **leading indicators** and **outcome metrics**
2. Proactively think about **unintended consequences** (guardrails)
3. Understand that **vanity metrics** (impressions, clicks) are not success metrics
4. Can connect a feature to **business value** (revenue, retention, expansion)

The rest of this notebook applies this framework to three realistic FlowDesk product scenarios.

---
## Section 2 — Feature 1: "Smart Daily Digest"

### Scenario
> The FlowDesk product team wants to ship a **daily email** that shows users a summary of what their teammates did yesterday — tasks completed, comments added, projects updated. The goal is to keep infrequent users engaged without requiring them to log in.

---

### Clarify First: Who Is This Feature For?
Before picking a metric, ask: *who is the target user?*

- **Not for daily active users** — they already log in and see this in-app
- **For infrequent users** — managers, stakeholders, part-time contributors who log in < 3x/week
- **Goal**: pull these users back in-app, or at minimum keep them informed enough to stay subscribed

---

### Applying the Framework

| Layer | Metric | Rationale |
|---|---|---|
| **Primary** | DAU among infrequent users (< 3 sessions/week) | Directly measures if the email pulled lapsed users back in-app |
| **Why NOT open rate** | Vanity — users can open and not engage | Open rate is a leading indicator, not an outcome |
| **Why NOT total DAU** | Too noisy — diluted by always-active users; feature effect invisible | |
| **Why NOT click rate** | Too narrow — a user might engage in-app without clicking the email | |
| **Guardrail 1** | Email unsubscribe rate must stay ≤ current baseline | Digest must not annoy users into unsubscribing from all FlowDesk emails |
| **Guardrail 2** | Average session duration | Digest should *pull users in-app*, not replace in-app time with email consumption |
| **Counter-metric 1** | "Mark as spam" rate | Hard negative signal — user actively rejects the email |
| **Counter-metric 2** | "Unsubscribe from all emails" rate | Catastrophic signal — feature caused user to opt out entirely |
| **Leading indicator** | Email open → in-app session within 4 hours | Moves in 24h; validates the email → re-engagement funnel is working |
| **Ecosystem** | In-app notification click rate | Does the digest cannibalize notification engagement? |

---

### Pre-Experiment Baseline Computation

Before running the experiment, we need **baseline values** for each metric. Without baselines, we cannot detect statistically significant changes. The cells below compute these from the actual FlowDesk dataset.

In [ ]:
# ── Baseline 1: Overall DAU ──────────────────────────────────────────────────
# DAU = distinct users with at least one event on a given day
events['event_date'] = events['event_ts'].dt.date

dau = (
    events.groupby('event_date')['user_id']
    .nunique()
    .reset_index()
    .rename(columns={'user_id': 'dau'})
)

dau['event_date'] = pd.to_datetime(dau['event_date'])

print("=== Overall DAU Summary ===")
print(f"  Date range  : {dau['event_date'].min().date()} → {dau['event_date'].max().date()}")
print(f"  Mean DAU    : {dau['dau'].mean():,.0f}")
print(f"  Median DAU  : {dau['dau'].median():,.0f}")
print(f"  Max DAU     : {dau['dau'].max():,.0f}")
print(f"  Min DAU     : {dau['dau'].min():,.0f}")

In [ ]:
# ── Baseline 2: Define "Infrequent Users" and compute their current activity ──
# Infrequent = logs in < 3x/week on average
# Use last 60 days of data as the baseline window

max_date = events['event_ts'].max()
baseline_start = max_date - pd.Timedelta(days=60)

baseline_events = events[events['event_ts'] >= baseline_start].copy()
baseline_events['week'] = baseline_events['event_ts'].dt.to_period('W')

# Sessions per user per week (distinct days with activity = proxy for sessions)
user_weekly_days = (
    baseline_events
    .groupby(['user_id', 'week'])['event_date']
    .nunique()
    .reset_index()
    .rename(columns={'event_date': 'active_days'})
)

# Average sessions per week across all weeks for each user
avg_weekly_sessions = (
    user_weekly_days
    .groupby('user_id')['active_days']
    .mean()
    .reset_index()
    .rename(columns={'active_days': 'avg_weekly_sessions'})
)

# Classify users
avg_weekly_sessions['user_type'] = np.where(
    avg_weekly_sessions['avg_weekly_sessions'] < 3,
    'Infrequent (<3x/week)',
    'Frequent (≥3x/week)'
)

type_counts = avg_weekly_sessions['user_type'].value_counts()
total_active = len(avg_weekly_sessions)

print("=== User Activity Segmentation (Last 60 Days) ===")
for utype, count in type_counts.items():
    print(f"  {utype:<30}: {count:,} users ({count/total_active*100:.1f}%)")

infrequent_users = avg_weekly_sessions[
    avg_weekly_sessions['user_type'] == 'Infrequent (<3x/week)'
]['user_id'].tolist()

print(f"\n  Target audience for Smart Digest: {len(infrequent_users):,} users")
print(f"  Avg weekly sessions (infrequent): "
      f"{avg_weekly_sessions[avg_weekly_sessions['user_type']=='Infrequent (<3x/week)']['avg_weekly_sessions'].mean():.2f} days/week")

In [ ]:
# ── Baseline 3: DAU for Infrequent Users (target segment) ────────────────────
infrequent_set = set(infrequent_users)

dau_infrequent = (
    baseline_events[baseline_events['user_id'].isin(infrequent_set)]
    .groupby('event_date')['user_id']
    .nunique()
    .reset_index()
    .rename(columns={'user_id': 'dau_infrequent'})
)
dau_infrequent['event_date'] = pd.to_datetime(dau_infrequent['event_date'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: DAU of infrequent users
ax = axes[0]
ax.fill_between(dau_infrequent['event_date'], dau_infrequent['dau_infrequent'],
                alpha=0.3, color='steelblue')
ax.plot(dau_infrequent['event_date'], dau_infrequent['dau_infrequent'],
        color='steelblue', linewidth=2)
baseline_mean = dau_infrequent['dau_infrequent'].mean()
ax.axhline(baseline_mean, color='crimson', linestyle='--', linewidth=1.5,
           label=f'Baseline mean: {baseline_mean:,.0f}')
ax.set_title('DAU — Infrequent Users (Primary Metric Baseline)', fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Distinct Active Users')
ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# Right: Notification/event rate (proxy for ecosystem impact)
ax2 = axes[1]
notif_events = baseline_events[baseline_events['event_type'].isin(
    ['notification_clicked', 'notification_viewed', 'dashboard_viewed']
)]
notif_rate = (
    notif_events.groupby('event_date').size()
    .reset_index(name='count')
)
notif_rate['event_date'] = pd.to_datetime(notif_rate['event_date'])
ax2.fill_between(notif_rate['event_date'], notif_rate['count'],
                 alpha=0.3, color='darkorange')
ax2.plot(notif_rate['event_date'], notif_rate['count'],
         color='darkorange', linewidth=2)
ax2.axhline(notif_rate['count'].mean(), color='crimson', linestyle='--',
            linewidth=1.5,
            label=f'Baseline mean: {notif_rate["count"].mean():,.0f}')
ax2.set_title('Daily Notification/Dashboard Events\n(Ecosystem Metric Baseline)',
              fontweight='bold')
ax2.set_xlabel('Date')
ax2.set_ylabel('Event Count')
ax2.legend()
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.suptitle('Feature 1: Smart Daily Digest — Pre-Experiment Baselines',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("\n=== Pre-Experiment Baseline Summary ===")
print(f"  Primary metric baseline   : {baseline_mean:,.0f} infrequent DAU/day")
print(f"  Ecosystem metric baseline : {notif_rate['count'].mean():,.0f} notification events/day")
print(f"\n  → These are the values we'd use as control-arm benchmarks in the A/B test.")

---
## Section 3 — Feature 2: "Project Templates"

### Scenario
> Users can now start a project using a **pre-built template** instead of from scratch. Templates include common workflows like "Software Sprint", "Marketing Campaign", and "Client Onboarding". The hypothesis is that templates reduce friction and accelerate time-to-value.

---

### Applying the Framework

| Layer | Metric | Rationale |
|---|---|---|
| **Primary** | Median time-to-first-task (hours/days after project creation) | Captures the "aha moment" acceleration — template users should add their first task faster |
| **Why NOT templates created** | Vanity — a user can create a template project and immediately delete it | |
| **Why NOT template adoption rate** | Intermediate metric — adoption doesn't mean the project was useful | |
| **Guardrail 1** | Project deletion rate | Templates should not create projects users immediately abandon | |
| **Guardrail 2** | Overall project creation rate | Templates must not cannibalize organic project creation | |
| **Counter-metric** | Template abandonment rate (project from template, zero tasks added ever) | Hard signal that template created false confidence but no real work | |
| **Leading indicator** | `project_created_from_template` event rate | Moves immediately; validates adoption before we can measure task creation timing | |
| **Ecosystem** | Onboarding checklist completion rate | Templates may shortcut the onboarding flow — is that good or bad? | |

In [ ]:
# ── Baseline: Time-to-First-Task ─────────────────────────────────────────────
# Compute: for each user, days between their first project_created event
# and their first task_created event

project_events = events[events['event_type'] == 'project_created'].copy()
task_events    = events[events['event_type'] == 'task_created'].copy()

# First project creation per user
first_project = (
    project_events
    .groupby('user_id')['event_ts']
    .min()
    .reset_index()
    .rename(columns={'event_ts': 'first_project_ts'})
)

# First task creation per user
first_task = (
    task_events
    .groupby('user_id')['event_ts']
    .min()
    .reset_index()
    .rename(columns={'event_ts': 'first_task_ts'})
)

# Merge and compute gap
ttft = first_project.merge(first_task, on='user_id', how='inner')
ttft['time_to_first_task_hrs'] = (
    (ttft['first_task_ts'] - ttft['first_project_ts'])
    .dt.total_seconds() / 3600
)

# Filter to reasonable range: 0 to 30 days
ttft_clean = ttft[
    (ttft['time_to_first_task_hrs'] >= 0) &
    (ttft['time_to_first_task_hrs'] <= 720)
].copy()

ttft_days = ttft_clean['time_to_first_task_hrs'] / 24

median_days = ttft_days.median()
mean_days   = ttft_days.mean()

print("=== Time-to-First-Task Baseline ===")
print(f"  Users with both project + task events : {len(ttft_clean):,}")
print(f"  Median time-to-first-task             : {median_days:.2f} days ({median_days*24:.1f} hours)")
print(f"  Mean time-to-first-task               : {mean_days:.2f} days")
print(f"  P25                                   : {ttft_days.quantile(0.25):.2f} days")
print(f"  P75                                   : {ttft_days.quantile(0.75):.2f} days")

In [ ]:
# ── Distribution of Time-to-First-Task ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: distribution (cap at 15 days for readability)
ax = axes[0]
plot_data = ttft_days[ttft_days <= 15]
ax.hist(plot_data, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(median_days, color='crimson', linewidth=2, linestyle='--',
           label=f'Median: {median_days:.1f} days')
ax.axvline(mean_days, color='darkorange', linewidth=2, linestyle=':',
           label=f'Mean: {mean_days:.1f} days')
ax.set_title('Distribution: Time-to-First-Task\n(capped at 15 days)', fontweight='bold')
ax.set_xlabel('Days from Project Created to First Task')
ax.set_ylabel('User Count')
ax.legend()

# Right: cumulative distribution — what % of users added task within N days?
ax2 = axes[1]
sorted_days = np.sort(ttft_days[ttft_days <= 30].values)
cdf = np.arange(1, len(sorted_days)+1) / len(sorted_days)
ax2.plot(sorted_days, cdf * 100, color='steelblue', linewidth=2.5)
for threshold, color in [(1, 'green'), (3, 'darkorange'), (7, 'crimson')]:
    pct = (ttft_days <= threshold).mean() * 100
    ax2.axvline(threshold, color=color, linestyle='--', alpha=0.7,
                label=f'Day {threshold}: {pct:.0f}% have added a task')
ax2.set_title('Cumulative: % Users Who Added First Task\nby Day N', fontweight='bold')
ax2.set_xlabel('Days from Project Created')
ax2.set_ylabel('Cumulative % of Users')
ax2.set_xlim(0, 15)
ax2.set_ylim(0, 100)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x)}%'))
ax2.legend(fontsize=9)

plt.suptitle('Feature 2: Project Templates — Primary Metric Baseline',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Counter-Metric: Project Abandonment Rate ──────────────────────────────────
# Projects where user never created a task = "template abandonment" analog

users_with_project = set(project_events['user_id'].unique())
users_with_task    = set(task_events['user_id'].unique())

# Users who created a project but NEVER created a task
abandoned = users_with_project - users_with_task
abandonment_rate = len(abandoned) / len(users_with_project)

print("=== Project Abandonment Baseline (Counter-Metric) ===")
print(f"  Users who created ≥1 project     : {len(users_with_project):,}")
print(f"  Users who also created ≥1 task   : {len(users_with_task & users_with_project):,}")
print(f"  Users who never created a task   : {len(abandoned):,}")
print(f"  Abandonment rate (baseline)      : {abandonment_rate:.1%}")
print(f"\n  → Hypothesis: Template users should have LOWER abandonment than {abandonment_rate:.1%}.")
print(f"    If templates INCREASE abandonment, the feature is a guardrail violation.")

---
## Section 4 — Feature 3: "Client Portal"

### Scenario
> Users can invite **external clients** (non-FlowDesk users) to view — but not edit — project status. The goal is to reduce the need for status-update emails and increase FlowDesk's perceived value to decision-makers outside the team.

---

### Strategic Framing: This Is a Growth Feature

The Client Portal is not just an engagement feature — it is a **viral growth mechanism**. When a FlowDesk user shows their client a polished project portal, the client experiences FlowDesk's value firsthand. This creates a **bottom-up sales motion**: clients who see the portal become prospects.

This changes the primary metric entirely.

---

### Applying the Framework

| Layer | Metric | Rationale |
|---|---|---|
| **Primary** | Workspace expansion (new seats added OR plan upgrade) among Client Portal users | Portal is a growth driver — success = expansion, not just engagement |
| **Why NOT portal views** | Engagement metric, not outcome — clients can view and never convert | |
| **Why NOT invites sent** | Leading indicator, not outcome — many invites, zero upgrades = failure | |
| **Guardrail 1** | Existing user engagement (tasks, comments, events per user) | Client-facing work shouldn't distract existing members from doing real work | |
| **Guardrail 2** | Security-related support ticket rate | External access creates security exposure — monitor for spike | |
| **Counter-metric** | Portal disable rate (enabled then disabled within 30 days) | Users who tried it and turned it off — strong signal of failed value delivery | |
| **Leading indicator** | `invite_sent` event rate (per workspace, per week) | Already tracked in events table — moves immediately | |
| **Ecosystem** | Plan upgrade rate across all workspaces | Does Client Portal correlate with upsell? | |

In [ ]:
# ── Leading Indicator: invite_sent event rate by plan tier ────────────────────
invite_events = events[events['event_type'] == 'invite_sent'].copy()

# Join with users to get plan tier
invite_with_plan = invite_events.merge(
    users[['user_id', 'plan']], on='user_id', how='left'
)

# Invites per user per plan (normalize by user count per plan)
plan_user_counts = users.groupby('plan')['user_id'].nunique().reset_index(name='user_count')
plan_invite_counts = invite_with_plan.groupby('plan').size().reset_index(name='invite_count')

plan_invite_rate = plan_invite_counts.merge(plan_user_counts, on='plan')
plan_invite_rate['invites_per_user'] = (
    plan_invite_rate['invite_count'] / plan_invite_rate['user_count']
)

plan_order = ['free', 'pro', 'business', 'enterprise']
plan_invite_rate['plan'] = pd.Categorical(
    plan_invite_rate['plan'], categories=plan_order, ordered=True
)
plan_invite_rate = plan_invite_rate.sort_values('plan')

print("=== invite_sent Event Rate by Plan Tier ===")
print(plan_invite_rate[['plan', 'invite_count', 'user_count', 'invites_per_user']]
      .to_string(index=False))

In [ ]:
# ── Ecosystem: Workspace Plan Upgrade Rate by Plan Tier ───────────────────────
# Proxy: users on 'free' or 'pro' → proportion on higher tiers in workspaces
ws_plan_dist = workspaces.groupby('plan').size().reset_index(name='workspace_count')
ws_plan_dist['pct'] = ws_plan_dist['workspace_count'] / ws_plan_dist['workspace_count'].sum() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: invites per user by plan
ax = axes[0]
colors = ['#90CAF9', '#42A5F5', '#1976D2', '#0D47A1']
bars = ax.bar(
    plan_invite_rate['plan'].astype(str),
    plan_invite_rate['invites_per_user'],
    color=colors[:len(plan_invite_rate)],
    edgecolor='white', linewidth=0.8
)
for bar, val in zip(bars, plan_invite_rate['invites_per_user']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_title('Leading Indicator: Invites Sent per User\nby Plan Tier', fontweight='bold')
ax.set_xlabel('Plan Tier')
ax.set_ylabel('Invites per User (Total)')

# Right: workspace plan distribution (ecosystem context)
ax2 = axes[1]
ws_plan_sorted = ws_plan_dist.sort_values('plan')
bars2 = ax2.bar(
    ws_plan_sorted['plan'],
    ws_plan_sorted['pct'],
    color=['#EF9A9A', '#EF5350', '#C62828', '#7B1FA2'][:len(ws_plan_sorted)],
    edgecolor='white', linewidth=0.8
)
for bar, val in zip(bars2, ws_plan_sorted['pct']):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax2.set_title('Workspace Plan Distribution\n(Ecosystem Baseline)', fontweight='bold')
ax2.set_xlabel('Plan Tier')
ax2.set_ylabel('% of Workspaces')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))

plt.suptitle('Feature 3: Client Portal — Leading Indicator & Ecosystem Baselines',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("\n=== Business Insight ===")
if 'enterprise' in plan_invite_rate['plan'].values:
    ent = plan_invite_rate[plan_invite_rate['plan']=='enterprise']['invites_per_user'].values[0]
    free_val = plan_invite_rate[plan_invite_rate['plan']=='free']['invites_per_user'].values[0] if 'free' in plan_invite_rate['plan'].values else 0
    print(f"  Enterprise users send {ent:.2f} invites/user vs {free_val:.2f} for free users.")
print("  Higher-tier users are already more likely to invite collaborators.")
print("  Client Portal gives them a polished interface to do so — likely accelerates upsell.")

---
## Section 5 — DAU/MAU Ratio: Product Stickiness

### What Is DAU/MAU?

**DAU/MAU** (also called the **stickiness ratio**) is one of the most widely used product health metrics at consumer and B2B companies:

$$\text{Stickiness} = \frac{\text{Daily Active Users}}{\text{Monthly Active Users}}$$

Interpretation:
- A ratio of **0.30** means users visit **~9 days per month** on average
- A ratio of **1.0** would mean every monthly user is also active every day (theoretical max)
- Facebook targets **~0.50**; TikTok is **~0.60+**; most B2B SaaS targets **0.20–0.40**

### Why Is This Different for B2B?

For a B2B project management tool like FlowDesk:
- **0.20** = users visit ~6 days/month → reasonable for async tools
- **0.30** = users visit ~9 days/month → strong engagement (like a weekday-only habit)
- **0.40+** = users visit daily → power users, deeply embedded

The critical insight: **a low DAU/MAU is not automatically bad for B2B** — a user might complete a weekly planning session and not need to return daily. The metric must be interpreted in context of the product's use case.

In [ ]:
# ── Compute DAU/MAU (Stickiness) by Month ────────────────────────────────────
events_2023 = events[
    (events['event_ts'] >= '2023-01-01') &
    (events['event_ts'] < '2024-01-01')
].copy()

events_2023['month']      = events_2023['event_ts'].dt.to_period('M')
events_2023['event_date'] = events_2023['event_ts'].dt.date

stickiness_rows = []

for month, grp in events_2023.groupby('month'):
    mau = grp['user_id'].nunique()
    # DAU for each day in this month
    daily_dau = grp.groupby('event_date')['user_id'].nunique()
    avg_dau   = daily_dau.mean()
    ratio     = avg_dau / mau if mau > 0 else np.nan
    stickiness_rows.append({
        'month': str(month),
        'mau': mau,
        'avg_dau': avg_dau,
        'stickiness': ratio,
        'days_per_month': ratio * 30
    })

stickiness_df = pd.DataFrame(stickiness_rows)

print("=== FlowDesk DAU/MAU Stickiness by Month (2023) ===")
print(stickiness_df[['month', 'avg_dau', 'mau', 'stickiness', 'days_per_month']]
      .to_string(index=False, float_format=lambda x: f'{x:.3f}' if x < 10 else f'{x:,.0f}'))

In [ ]:
# ── Stickiness by Plan Tier ───────────────────────────────────────────────────
events_plan = events_2023.merge(
    users[['user_id', 'plan']], on='user_id', how='left'
)

plan_stickiness = []
for (plan, month), grp in events_plan.groupby(['plan', 'month']):
    mau     = grp['user_id'].nunique()
    avg_dau = grp.groupby('event_date')['user_id'].nunique().mean()
    plan_stickiness.append({
        'plan': plan,
        'month': str(month),
        'stickiness': avg_dau / mau if mau > 0 else np.nan
    })

ps_df = pd.DataFrame(plan_stickiness)
avg_by_plan = ps_df.groupby('plan')['stickiness'].mean().reset_index()
plan_order = ['free', 'pro', 'business', 'enterprise']
avg_by_plan['plan'] = pd.Categorical(avg_by_plan['plan'],
                                      categories=plan_order, ordered=True)
avg_by_plan = avg_by_plan.sort_values('plan').dropna()

print("=== Average Stickiness by Plan Tier (2023) ===")
for _, row in avg_by_plan.iterrows():
    days = row['stickiness'] * 30
    print(f"  {row['plan']:<12}: DAU/MAU = {row['stickiness']:.3f}  → ~{days:.1f} days/month")

In [ ]:
# ── Stickiness Charts ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Stickiness trend over 2023
ax = axes[0]
ax.plot(stickiness_df['month'], stickiness_df['stickiness'],
        marker='o', linewidth=2.5, markersize=7, color='steelblue')
ax.fill_between(range(len(stickiness_df)),
                stickiness_df['stickiness'],
                alpha=0.15, color='steelblue')
ax.axhline(0.30, color='green', linestyle='--', linewidth=1.5,
           label='B2B strong threshold (0.30)')
ax.axhline(0.20, color='darkorange', linestyle='--', linewidth=1.5,
           label='B2B baseline threshold (0.20)')
ax.set_xticks(range(len(stickiness_df)))
ax.set_xticklabels(stickiness_df['month'], rotation=45, ha='right')
ax.set_title('DAU/MAU Stickiness Trend — 2023', fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('DAU / MAU Ratio')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.2f}'))
ax.legend(fontsize=9)

# Right: Stickiness by plan tier (bar)
ax2 = axes[1]
colors_plan = ['#90CAF9', '#42A5F5', '#1976D2', '#0D47A1']
bars = ax2.bar(
    avg_by_plan['plan'].astype(str),
    avg_by_plan['stickiness'],
    color=colors_plan[:len(avg_by_plan)],
    edgecolor='white', linewidth=0.8
)
for bar, val in zip(bars, avg_by_plan['stickiness']):
    ax2.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.003,
             f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax2.axhline(0.30, color='green', linestyle='--', linewidth=1.5,
            label='B2B strong (0.30)')
ax2.set_title('Average DAU/MAU by Plan Tier (2023)', fontweight='bold')
ax2.set_xlabel('Plan Tier')
ax2.set_ylabel('DAU / MAU (Annual Average)')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.2f}'))
ax2.legend(fontsize=9)

plt.suptitle('FlowDesk Product Stickiness — DAU/MAU Analysis',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### Business Interpretation

**What does a DAU/MAU of ~0.30 mean for FlowDesk?**

- Users visit approximately **9 days per month** on average
- For a B2B project management tool, this is **healthy** — it implies a weekday-usage habit (5 days/week = 22 days/month; once every 2-3 days = 10-15 days)
- Enterprise users being stickier makes intuitive sense: they are more deeply embedded, have more teammates on the platform, and have longer contracts that create behavioral lock-in

**Is 0.30 good or bad?**

| Context | DAU/MAU | Assessment |
|---|---|---|
| Social media (Facebook) | 0.50+ | Expected — daily habit |
| Video/content (TikTok) | 0.60+ | Highly habitual |
| B2B SaaS (productivity) | 0.20–0.40 | Normal range |
| FlowDesk (current) | ~0.28–0.33 | **Healthy for B2B** |
| B2B concern threshold | < 0.15 | Users forget the product exists |

**The enterprise gap is a product signal:** If enterprise users have significantly higher stickiness than free users, it suggests the product delivers more value at depth — and that the path to improving free-tier stickiness is onboarding users into the behaviors that enterprise users already have (collaborative features, recurring workflows, integrations).

---

## Summary — What This Notebook Demonstrates

| Skill | Evidence |
|---|---|
| Metric framework thinking | 5-layer framework applied to 3 distinct feature types |
| Distinguishing primary vs. vanity metrics | Explained why open rate, views, and template counts are wrong |
| Baseline computation | DAU, time-to-first-task, invites/user, stickiness all computed from real data |
| Business strategy connection | Client Portal framed as a growth/expansion driver, not just engagement |
| DAU/MAU interpretation | Ratio contextualized by B2B norms and plan-tier segmentation |

> **Interview takeaway:** The best metric frameworks are not chosen from a menu — they are derived from the product goal, the target user, and the business model. The framework in this notebook provides a structure to derive the right metric under time pressure.